In [1]:
!pip install -q transformers torch flask pyngrok peft bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.8 MB/s eta 0:00:00


In [2]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 116.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [3]:
import gc
import threading
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from flask import Flask, request, jsonify
from pyngrok import ngrok
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
BASE_MODEL_NAME = "Qwen/Qwen3.5-4B"

BUSINESS_ADAPTER_PATH = "/content/drive/MyDrive/adapters/business_adapter"
LEGAL_ADAPTER_PATH    = "/content/drive/MyDrive/adapters/laws_adapter"

NGROK_TOKEN = "39iAlJPt1mmeQ684s5ReWT5nrr7_25Mbm5f8qS8YjykTvwCdZ"
MAX_CONTEXT_TOKENS = 10000

In [5]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print("Загрузка токенайзера")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

print("Загрузка базовой модели (4-bit)")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)

print("Загрузка адаптера business")
model = PeftModel.from_pretrained(base_model, BUSINESS_ADAPTER_PATH, adapter_name="business")

print("Загрузка адаптера legal")
model.load_adapter(LEGAL_ADAPTER_PATH, adapter_name="legal")

model.eval()
print("Модель и оба адаптера загружены")

Загрузка токенайзера


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Загрузка адаптера business
Загрузка адаптера legal
Модель и оба адаптера загружены


In [6]:
SYSTEM_PROMPTS = {
    "business": """
Ты отвечаешь пользователю как опытный предприниматель с многолетним практическим опытом ведения бизнеса и управления компаниями.

Характер речи и тон:
- Чёткие и уверенные формулировки.
- Ориентация на практический результат: прибыль, эффективность, рост бизнеса.
- Объяснения простым и понятным языком без лишней теории.
- Использование деловой лексики: стратегия, масштабирование, юнит-экономика, рынок, конкурентные преимущества.

Структура ответов:
1. Краткий основной вывод или позиция.
2. Практическое объяснение, почему это работает или важно.
3. Конкретные действия или рекомендации (3–7 пунктов).
4. Краткое итоговое замечание или стратегический совет.

Запрещено: давать советы по незаконной деятельности, уходить в абстрактную теорию без практической пользы.
Если вопрос не напрямую связан с бизнесом — рассматривай его через призму карьеры, эффективности, управления ресурсами или предпринимательского мышления.
""",

    "legal": """
Ты отвечаешь пользователю как опытный юрист-практик с многолетним стажем юридической работы.

Характер речи:
- Чёткие, точные и понятные формулировки.
- Объяснение правовых принципов простым языком.
- Фокус на практических рекомендациях и реальных действиях.
- Уважительное и профессиональное общение.

Структура ответов:
1. Прямой ответ на вопрос.
2. Краткое объяснение правовых принципов или норм, которые применяются.
3. Практические шаги, которые стоит предпринять (3–7 пунктов).
4. Возможные риски и на что обратить внимание.

Запрещено: советовать незаконные действия, давать вводящие в заблуждение рекомендации или игнорировать возможные риски.
Если вопрос не юридический — рассматривай его через призму правовых рисков, защиты интересов и юридической грамотности.
""",
}

# Блокировка нужна если переключение адаптера — операция изменения состояния модели.
# При threaded=False в Flask запросы обрабатываются последовательно, но lock — страховка на будущее.
adapter_lock = threading.Lock()

In [12]:
app = Flask(__name__)


@app.route("/generate", methods=["POST"])
def generate():
    try:
        data = request.json
        if not data:
            return jsonify({"error": "Нет данных", "status": "error"}), 400

        user_message = data.get("message", "")
        history = data.get("history", [])
        consultant = data.get("consultant", "business")

        if not user_message:
            return jsonify({"error": "Пустое сообщение", "status": "error"}), 400

        # Безопасный fallback если передан неизвестный тип
        if consultant not in SYSTEM_PROMPTS:
            consultant = "business"

        system_prompt = SYSTEM_PROMPTS[consultant]

        # Сборка контекста с учётом лимита токенов
        messages = [{"role": "system", "content": system_prompt}]

        system_tokens      = len(tokenizer.encode(system_prompt))
        user_message_tokens = len(tokenizer.encode(user_message))
        current_tokens     = system_tokens + user_message_tokens

        for msg in reversed(history):
            msg_tokens = len(tokenizer.encode(msg.get("content", "")))
            if current_tokens + msg_tokens < MAX_CONTEXT_TOKENS:
                messages.insert(1, {"role": msg["role"], "content": msg["content"]})
                current_tokens += msg_tokens
            else:
                break

        messages.append({"role": "user", "content": user_message})

        # Переключение адаптера и генерация
        with adapter_lock:
            model.set_adapter(consultant)

            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
            model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

            with torch.no_grad():
                generated_ids = model.generate(
                    **model_inputs,
                    max_new_tokens=800,
                    temperature=0.7,
                    top_p=0.9,
                    do_sample=True,
                    pad_token_id=tokenizer.eos_token_id,
                )

            generated_ids = [
                output[len(inp):]
                for inp, output in zip(model_inputs.input_ids, generated_ids)
            ]
            response_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

            del model_inputs, generated_ids

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        return jsonify({"response": response_text, "status": "success"})

    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({"error": str(e), "status": "error"}), 500


@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "model": BASE_MODEL_NAME,
                    "adapters": ["business", "legal"]})


@app.route("/", methods=["GET"])
def index():
    return jsonify({"message": "AI Consultant API", "status": "running",
                    "consultants": list(SYSTEM_PROMPTS.keys())})

In [ ]:
ngrok.kill()
ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(5000)

print(f"\n{'='*50}")
print(f"URL:      {public_url}")
print(f"Endpoint: {public_url}/generate")
print(f"Health:   {public_url}/health")
print(f"{'='*50}\n")

app.run(host="0.0.0.0", port=5000, threaded=False)



URL:      NgrokTunnel: "https://indictable-soothly-samira.ngrok-free.dev" -> "http://localhost:5000"
Endpoint: NgrokTunnel: "https://indictable-soothly-samira.ngrok-free.dev" -> "http://localhost:5000"/generate
Health:   NgrokTunnel: "https://indictable-soothly-samira.ngrok-free.dev" -> "http://localhost:5000"/health

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [10/Mar/2026 22:49:52] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Mar/2026 22:49:52] "GET /health HTTP/1.1" 200 -
